# Lecture 32 - "Where" questions with GeoPandas
### EES 3350/5350 · Python in Earth Science · Friday, November 20, 2026 

Earlier, you **made** maps. Today you'll ask questions with them: which flood zone is a gauge in? Which rock unit is undert each earthquake hypocenter? How many earthquakes happend within 50 km of Nisqually?

We're going back to the Puget Sound, where the 2001 Nisqually earthquake shook Seattle, Tacoma, and Olympia. In Assignment 2, you plotted its after shocks on a blank set of axes. Today they go on a real map, with 26 years of other earthquakes and the geology underneath them.

**Learning Objectives**
- turn a table with latitude and longitude columns into a GeoDataFrame
- explore a map interactively, and connect what you click to the row in the table
- look inside the `geometry` column: what a shape actually *is*
- change how big a map looks: color, size, and basemap
- **clip** a big map to the area you care about
- use a **spatial join** to match rows by *where* they are, then summarize within `groupby`
- draw a **buffer** (a circle of a given radius) and find what's inside it
- subtract one set of shapes form another with an **overlay**

*Reminder that proposals are due at 5pm today*

---
## Part 0 : set up

In [ ]:
%pip install folium mapclassify

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as cx

maps = "../../course-materials/week13/maps/"

---
## Part 1 : use `.explore` to quickly look at data

One easy way to quickly explore data is to use the `.explore` method on your GeoDataFrame.
- .`explore` makes a **web map** that you can zoom, drag, and click. Every point still remembers its row in the table.

#### First, let's import our gauges and floodplain data, and change the latter's CRS to match that of the former

In [ ]:
gauges = gpd.read_file(maps + "gauges.geojson")
floodplain = gpd.read_file(maps + "davidson_floodplain.zip")
floodplain = floodplain.to_crs(gauges.crs)
floodplain.head()

Just a reminder that `floodplain` is a GeoDataFrame with the 100- and 500-year floodplains. 

> **Note** that these are now `MULTIPOLYGON` data, which means they have multiple different polygons within them

#### Now, let's use `.explore`.

Below is a cell where each line slowly adds different things. Play around!

In [ ]:
floodplain.explore()                                                                                  # nothing added
#floodplain.explore(column="zone", cmap="viridis")                                                     # change the color based on what zone it is
#floodplain.explore(column="zone", cmap="viridis", tiles=cx.providers.Esri.WorldGrayCanvas)            # change the background map
#floodplain.explore(column="zone", cmap="viridis", tiles=cx.providers.Esri.WorldShadedRelief)          # change the background map again
#floodplain.explore(column="zone", cmap="viridis", tiles=cx.providers.Esri.WorldTopoMap, height=750)   # change the background map again and also the size of the map

### What if we want to look at two different datasets at the same time?

We can do that the same way we did with `ax=ax`. `.explore` gives back the map, you just pass that map to the next layer using `m=`

In [ ]:
m = floodplain.explore(column="zone", cmap="viridis", tiles=cx.providers.Esri.WorldTopoMap) # this is what we did above, but we just set it equal to some variable
gauges.explore(m=m, color="red", marker_kwds={"radius": 8}, tooltip=["short_name"]) # we then use that variable as `m=` (it's `m` because it's a map)
m

- `marker_kwds` is a dictionary that can help us set characteristics of the `marker_type`
    - `{"radius": 8}` sets the radius of the marker
    - `{"fill": ____}` whether to fill the `circle` or `circle_marker` with color
- `tooltip` controls what GoeDataFrame attributes are displayed when you hover over an object
    - `True` includes all columns
    - `False` removes the tooltip
    - You can also pass it a string or list of strings to specify a column (like we did)
    - If you pass it an integer, it specifices the firs n columns to be included 

### Now we can move on to asking questions about our data, like: ***Which flood zone is each gauge in?***

---
## Part 2 : combining data (i.e., spatial joins)
To answer this question, we need to merge some things*. In Lecture 25, you **merged** two tables on a shared key: `pd.merge(a, b, on="data")`. A **spatial join** (`gpd.sjoin`) is the same idea, except the "key" is **where** each row is. It answers questinos like "*which polygon is each point inside?*"

|**join**|**matches a row in A with a row in B when...**|
|:--|:--|
|`pd.merge(a, b, on="id")`| they have the same `id`|
|`gpd.sjoin(a, b, predicate="within")`|A's shape is **within** B's shape|

**yes, I know technically we could just look at the map, but we want to do this using code*

### Let's combine `gauges` and `floodplain` using `.sjoin`

### What `.sjoin` does:
Imagine you're holding a printed map of the flood zones and list of the three gauges. For each gauge, you put your finger on it and ask: **which zone am I in?** Then you'd write that zone next ot the gauges' name on your list.

That's a **spatial join**. `gpd.sjoin()` does the finger-on=the-map part for every row, and writes the answer into the table for you.

#### **Before:** We have two separate tables
|`gauges`| |`floodplain`||
|:--|:--|:--|:--|
|**short_name**|**geometry**|**zone**|**geometry**|
|Cumberland|POINT|100-year| POLYGON (all the 100-year areas)|
|Harpeth|POINT|500-year| POLYGON (all the 500-year areas)|
|Mill Creek| POINT| | |

**What `sjoin` does, row by row:**
1. Take the first gauge (Cumberland). Check each flood-zone polygon: is this point **within** it?
2. It's inside the 100-year polygon, so copy that polygon's columns (`zone`) onto the Cumberland's row.
3. Repeat for every gauge. The Harpeth isn't inside any polygon, so it gets `NaN`.

#### **After:** We have one table
|**short_name**|**geometry** |**index_right**|**zone**|
|:--|:--|:--|:--|
|Cumberland|POINT|0| 100-year|
|Harpeth|POINT|NaN| NaN |
|Mill Creek| POINT|0 |100-year |

The result keeps the **left** table's rows and shapes (the gauges), and gains to the **right** tables columns. `index_right` tells you which row of `floodplain` each gauge matched (row 0 is the 100-year zone).

In [ ]:
gauges = gpd.read_file(maps + "gauges.geojson")
floodplain = gpd.read_file(maps + "davidson_floodplain.zip")

floodplain = floodplain.to_crs(gauges.crs)

gauges_zones = gpd.sjoin(gauges, floodplain, predicate="within", how="left") # NEW
gauges_zones[["short_name", "zone"]]

### Reading the `sjoin` line

`gpd.sjoin(gauges, floodplain, predicate="within", how="left")`

| piece | what it means |
|:--|:--|
| `gauges` (first) | the **left** table: you get one row per gauge back, and keep the gauges' points |
| `floodplain` (second) | the **right** table: its columns get copied over |
| `predicate="within"` | the test for a match: is the left shape **inside** the right shape? |
| `how="left"` | keep every gauge, even ones with no match (they get `NaN`) |

**It's `merge` with a different key.**

| | `pd.merge(a, b, on="date")` | `gpd.sjoin(a, b, predicate="within")` |
|:--|:--|:--|
| rows match when... | they have the **same date** | A's shape is **inside** B's shape |
| `how="left"` | keep every row of A | keep every row of A |
| you get | A's columns + B's columns | A's columns + B's columns |

**Other tests you can use for `predicate=`:**

| `predicate=` | matches when the left shape... | example |
|:--|:--|:--|
| `"within"` | is completely inside the right shape | which watershed is each gauge in? |
| `"intersects"` | touches or overlaps the right shape at all | which counties does a river flow through? |
| `"contains"` | completely surrounds the right shape | which basins contain at least one gauge? |

### Okay, but if you notice on the `.explore()` map, the floodplain includes lakes...let's figure out how to remove them.

The `davidson_water` shapefile holds the Census Bureau's lakes, reservoirs, and rivers for the county.

##### Let's read that file...

In [ ]:
water = gpd.read_file(maps + "davidson_water/davidson_water.shp")

##### ...convert it to the right CRS (let's choose UTM)...

In [ ]:
water_utm = water.to_crs(epsg=32616)
floodplain_utm = floodplain.to_crs(epsg=32616)

##### ...and plot the ater on top of the flood zones:

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))

floodplain_utm.plot(ax=ax, column="zone", cmap="Blues_r", legend=True)
water_utm.plot(ax=ax, color="pink", tiles=cx.providers.Esri.WorldTopoMap)

#### That's a lot of water plotted as the floodplain! Let's subtrack the water from the flood zones using `gpd.overlay()`

### What `.overlay` does:
`sjoin` **copies columns** from one table to another, and never changes the shapes. `overlay` **cuts the shapes themselves**, like laying two sheets of tracing paper on top of each other an cutting along every line where they cross.

Because FEMA's 100-year zone includes water like Percy Priest Lake, Old Hickory Lake, and the Cumberland River itself, we want to find **only the ***land*** that floods**. We need to cut the water out of the flood zones.

#### **Before:** We have two layers that overlap
|`water`| |`floodplain`||
|:--|:--|:--|:--|
|**FULLNAME**|**geometry**|**zone**|**geometry**|
|Percy Priest Reservoir|POLYGON|100-year| POLYGON (all the 100-year areas)|
|Cumberland River|POLYGON|500-year| POLYGON (all the 500-year areas)|
|Old Hickory Lk| POLYGON| | |
|...(64 water bodies)| | |

**What `.overlay(floodplain, water, how="difference")` does:**
1. Lays the water polygons on top of the flood-zone polygons.
2. Wherever they overlap, **cuts the water out** of the flood zone, like a cookie cutter.
3. Keep what's left of the flood zones.

#### **After:** We have the same two zones, with lake- and river-shaped holes
|**zone**|**geometry** |**area**|
|:--|:--|:--|
|100-year|POLYGON, with holes where the water was|175 -> **120 km$^2$**|
|500-year|POLYGON (barely changed)|23 -> **23 km$^2$**|


#### Let's do it:

In [ ]:
fig, ax = plt.subplots(figsize=(20, 20))

floodplain_utm.plot(ax=ax, column="zone", cmap="Blues_r", legend=True)
water_utm.plot(ax=ax, color="pink", tiles=cx.providers.Esri.WorldTopoMap)
land = gpd.overlay(floodplain_utm, water_utm, how="difference")

### Reading the `overlay` line
`gpd.overlay(floodplain_utm, water_utm, how="difference")`

| piece | what it means |
|:--|:--|
| `floodplain_utm` (first) | layer **A**: the shapes being cut |
| `water_utm` (second) | layer **B**: the cookie cutter |
| `how="difference"` | which pieces to keep: A **minus** B |

**`how=` is set math for shapes**. Picture two overlapping circles, A & B: 

| `how=`|keeps| Nashville example | columns you get |
|:--|:--|:--|:--|
| `"difference"` | A, minus where B overlaps it | flood zones that are **land**| A's only |
| `"intersection"` | only where A and B overlap | flood zones that are **water** | A's **and** B's|
| `"union"` | everything, cut into pieces wherever they cross | every piece of flood zone and water, labeled | A's **and** B's |
|`"symmetric_difference"`| A or B, but not both| flood zones that aren't water, plus water outside of the flood zones | A's **and** B's|

**Three shape tools, side by side:**

| tool | changes the shape? | columns you get | use it for |
|:--|:--|:--|:--|
| `gpd.sjoin(a, b)` | no| A's + B's | which polygon is each point in? |
| `gpd.clip(a, b)` |yes: cut A to B's outline| A's only | trim a big map to your area |
| `gpd.overlay(a, b, how=)` |yes: cut both along every crossing| depends on `how` | subtract, overlap, or combine areas |


<div class="alert alert-danger">

**Common Error:** Running `overlay` on layers in different CRS, or in degrees. Liek `sjoin` both layers must have the **same CRS**, and if you're going to measure area afterward, convert both to a projected CRS (like UTM) **first**.
    
</div>

In [ ]:
land = gpd.overlay(floodplain_utm, water_utm, how="difference")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

floodplain_utm.plot(ax=axes[0], column="zone", cmap="Blues_r", tiles=cx.providers.Esri.WorldTopoMap)
axes[0].set_title("A: FEMA flood zones")

water_utm.plot(ax=axes[1], color="black", tiles=cx.providers.Esri.WorldTopoMap)
axes[1].set_title("B: lakes and rivers")

land.plot(ax=axes[2], column="zone", cmap="Blues_r", tiles=cx.providers.Esri.WorldTopoMap)
axes[2].set_title('overlay(A, B, how="difference")')

for ax in axes:
    ax.set_axis_off()

### The 100-year zone drops from about 175 km$^2$ to **120 km$^2$ of land, 8.8% of the county** (from 12.9%).
> Almost all of the difference is the open water of Percy Priest, Old Hickory Lake, and the Cumberland itself.
>
> The 500-year zone barely changes, because it's land beyond the 100-year zone. `overlay` is the shape version of set math: `"difference"` keeps A minus B, `"intersection"` keeps only where they overlap, and `"union"` keeps both.

---
## Part 3 : points from a table

We're going to go to the Puget Sound for the rest of class, thinking about earthquakes. Earthquake catalogues, sample sites, and field stations usually come as a **CSV** with a latitude column and a longitude column, not as a map file. Here's teh Nisqually file from Lab 06:

In [ ]:
!head -9 ../../course-materials/week13/maps/nisqually_2001.csv

#### Let's extract the column names and then read in the file, using those column names.

In [ ]:
column_names = ["hours", "latitude", "longitude", "depth", "mag", "is_earthquake"]
nisqually = pd.read_csv(maps + "nisqually_2001.csv", comment="#", header=None, names=column_names)

nisqually.head()

That's an ordinary DataFrame: the numbers are there, but pandas doesn't know that they're places. 

#### Two steps turn it into a GeoDataFrame:
- `gpd.points_from_xy(longitude, latitude)` builds one `POINT` per row
    - **first argument** : the east-west column (longitude)
    - **second argument** : the north-south column (latitude) 

This makes it so that `gpd.GeoDataFrame(..)` can attach those points as the `geometry` column. Because there's no CRS built into the data, you need to explicitly define it using `crs=` (in this case `crs=EPSG:4326`.

In [ ]:
points = gpd.points_from_xy(nisqually["longitude"], nisqually["latitude"])

nisqually = gpd.GeoDataFrame(nisqually, geometry=points, crs="EPSG:4326")

nisqually.head()

#### Now it's a **GeoDataFrame!**

<div class="alert alert-danger">

**Common Error**: Putting latitude first. People say "lat, long", but maps are plotted **x**, **y**, so it's `points_from_xy(longitude, latitude)`. If your Washington earthquakes show up in Antarctica (latitude 122), you swapped them.
    
</div>

The second file is every earthquake magnitude of 2 or larger in the Puget sound region from 2000 through 2025, straight from the USGS catalog. It's a normal ComCat download, so its columns already have names.

<span style="font-size:0.85em;color:#6b7a8d;"><em>Data: U.S. Geological Survey, ANSS Comprehensive Catalog (ComCat), downloaded November 2026: <code>https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&starttime=2000-01-01&endtime=2026-01-01&minlatitude=46.8&maxlatitude=48.3&minlongitude=-123.3&maxlongitude=-121.7&minmagnitude=2&eventtype=earthquake</code></em></span>

### Let's explore the depth of the Nisqually earthquake data using `.explore()`

In [ ]:
nisqually.explore(column="depth", cmap="viridis")

<div class="alert alert-warning">

**Question**: Where were the deepest earthquakes? What about the shallowest ones?
    
</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

The deepest earthquakes are in the southern Puget Sound, between Anderson and McNeil Islands. The deepest ones occur in evern further south, just eat of Centralia and Chehalis.

<div class="alert alert-info">

### Exercise 1 : the Puget Sound catalog as a GeoDataFrame

1. Read `puget_sound_quakes.csv` with `pd.read_csv()`. How many earthquakes are there? Which columns hold the locaiton?
2. Turn it into a GeoDataFrame called `quakes`, in EPSG:4326.
3. Plot it, colored by `depth`, with a legend.
    
</div>

In [ ]:
# 1
quakes = pd.read_csv(maps + _____")
print(len(quakes), "earthquakes")

# 2
points = gpd.points_from_xy(quakes["____"], quakes["____"])
quakes = gpd.GeoDataFrame(quakes, geometry=____, crs="____")

# 3
fig, ax = plt.subplots(figsize=(6,6))
quakes.plot(ax=ax, column=____, cmap=_____, markersize=5, legend=True, tiles=cx.providers.Esri.WorldTopoMap)
ax.set_title("Puget Sound earthquakes, M2+, 2000-2025 (color = depth, km)");

### Explore this data
We can use `.explore()` to zoom around and see information about each of these earthquakes. Below, we use `tooltip` as a parameter to filter the information that we see when we hover over a point.

In [ ]:
quakes.explore(column="depth", cmap="viridis",
               tooltip=["id", "time", "mag", "depth"],
               tiles=cx.providers.Esri.WorldGrayCanvas,
               #width=700, height=450
              )

<div class="alert alert-info">

### Exercise 2 : click, then find the row
1. Using the map, what is the `id` of the **deepest** earthquake? Write the `id` below in the appropriate place to pull that row out of the table with a mask.
2. Then, let pandas find it for you: use a mask to get the row where `depth` equals the **maximum** depth. Did you find the same one?
    
</div>

In [ ]:
# your answer here
visual_deepest = quakes[quakes["id"] == "____"]
visual_id = visual_deepest["____"].iloc[0]
visual_depth = visual_deepest["____"].iloc[0]
print(f"From looking at the map, the deepest earthquake is ID: {visual_id}, which is at {visual_depth} m depth.")

pandas_deepest = quakes[quakes["depth"] == quakes["depth"].____()]
pd_id = pandas_deepest["____"].iloc[0]
pd_depth = pandas_deepest["____"].iloc[0]
print(f"\nUsing pandas, the deepest earthquake is ID: {pd_id}, which is at {pd_depth} m depth.")

#### It should be the same because it's the same data.
The deepest earthquake in the catalog is 64 km down, deeper than Nisqually.

<center>
    <b>
        <span style="font-size:25px">This is Assignment 2, but with a real map behind it!</span></b></center>
There are 1,112 earthquakes. Most are 10-30 km deep, and the deepest ones (yellow, 40-64 km) are in the south, like Nisqually. Those deep earthquakes break inside the Juan de Fuca plate as it sicks under Washington state.

#### Let's finish the figure by adding the Nisqually main shock and aftershocks. 
Let's first convert everything to Web Mercator (EPSG:3857), because that's what the aftershock and main shock data are in right now.

### How am I filtering out the main shock and aftershock? What python/pandas/GeoPandas tool am I using below?

In [ ]:
quakes_web = quakes.to_crs(epsg=3857)
nisqually_web = nisqually.to_crs(epsg=3857)

mainshock = nisqually_web[nisqually_web["hours"] == 0]
aftershocks = nisqually_web[(nisqually_web["hours"] > 0) & (nisqually_web["is_earthquake"] == 1)]

fig, ax = plt.subplots(figsize=(7, 8))
quakes_web.plot(ax=ax, color="gray", markersize=4, alpha=0.5, label="M2+ earthquakes, 2000-2025")
aftershocks.plot(ax=ax, color="red", markersize=25, label="Nisqually aftershocks (first week)")
mainshock.plot(ax=ax, color="gold", edgecolor="black", marker="*", markersize=400, label = "Nisqually mainshock")

cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
ax.set_axis_off()
ax.legend(loc="lower left")
ax.set_title("The 2001 Nisqually earthquake and\n 26 years of Puget Sound earthquakes");

### Masks!
We used the same two masks from Lab 06 (`hours > 0`, `is_earthquake ==1`), which work on a GeoDataFrame (thank gawd). Nisqually had very few aftershocks for an M6.8, a pattern of deep earthquakes, and the gray dots show the Puget Sound shakes all the time anyways.

### We can also plot the Cedar River on this map!

In [ ]:
nldi = "https://api.water.usgs.gov/nldi/linked-data/nwissite/"

def flowlines(site, mode, km):
    """River lines from the NLDI. mode: 'UM' upstream main, 'UT' upstream + tributaries, 'DM' downstream main."""
    return gpd.read_file(f"{nldi}{site}/navigation/{mode}/flowlines?distance={km}")

In [ ]:
cedar_sites = {"Renton": "USGS-12119000", "Landsburg": "USGS-12117500"}

pieces = []
for name, site in cedar_sites.items():
    g = gpd.read_file(nldi + site)
    g["short_name"] = name
    pieces.append(g)
cedar_points = pd.concat(pieces, ignore_index=True)

cedar_basin = gpd.read_file(nldi + cedar_sites["Renton"] + "/basin")
cedar_tribs = flowlines(cedar_sites["Renton"], "UT", 100)
cedar_main = flowlines(cedar_sites["Renton"], "UM", 100)

web = 3857   # Web Mercator, the projection basemap tiles use

cedar_basin_w = cedar_basin.to_crs(epsg=web)
cedar_tribs_w = cedar_tribs.to_crs(epsg=web)
cedar_main_w = cedar_main.to_crs(epsg=web)
cedar_points_w = cedar_points.to_crs(epsg=web)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))

cedar_basin_w.plot(ax=ax, color="#009E73", alpha=0.12, edgecolor="#009E73")
cedar_main_w.plot(ax=ax, color="#0072B2", linewidth=2.5, label="Cedar River")

quakes_web = quakes.to_crs(epsg=3857)
nisqually_web = nisqually.to_crs(epsg=3857)

mainshock = nisqually_web[nisqually_web["hours"] == 0]
aftershocks = nisqually_web[(nisqually_web["hours"] > 0) & (nisqually_web["is_earthquake"] == 1)]

quakes_web.plot(ax=ax, color="gray", markersize=4, alpha=0.5, label="M2+ earthquakes, 2000-2025")
aftershocks.plot(ax=ax, color="red", markersize=25, label="Nisqually aftershocks (first week)")
mainshock.plot(ax=ax, color="gold", edgecolor="black", marker="*", markersize=400, label = "Nisqually mainshock")

cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
ax.set_axis_off()
ax.legend(loc="lower left")
ax.set_title("The 2001 Nisqually earthquake and\n 26 years of Puget Sound earthquakes");

---
## Part 3 : make it yours

`.plot()` has more knobs than we've used. A fun one is `markersize`: it can be a **column**, so every point can get it's own size depending on that column's value.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

quakes_web = quakes.to_crs(epsg=3857)
nisqually_web = nisqually.to_crs(epsg=3857)

mainshock = nisqually_web[nisqually_web["hours"] == 0]
aftershocks = nisqually_web[(nisqually_web["hours"] > 0) & (nisqually_web["is_earthquake"] == 1)]

quakes_web.plot(ax=ax, 
                column="depth",
                cmap="viridis", 
                markersize=quakes["mag"]**3, # We edited this
                alpha=0.6,
                legend=True
               )
aftershocks.plot(ax=ax, color="red", markersize=25, label="Nisqually aftershocks (first week)")
mainshock.plot(ax=ax, color="gold", edgecolor="black", marker="*", markersize=400, label = "Nisqually mainshock")

cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
ax.set_axis_off()
ax.legend(loc="lower left")
ax.set_title("The 2001 Nisqually earthquake and\n 26 years of Puget Sound earthquakes");

- `.markersize=quakes["mag"]**3` gives each point an area that grows with its magnitude:
  - an M2 gets 8
  - an M3 gets 27
  - Nisqually (M6.8) gets 314

Remember that magnitude is a log scale, so even cubing it **underestimates** how much bigger Nisqually was.

<div class="alert alert-info">

### Exercise 3 : color by year

Did the Puget Sound get more earthquakes over time, or were they spread evenly? Make a `year` column, then color the map by it instead of by depth. Try a different colormap than your neighbors (find other colormaps [here](https://matplotlib.org/stable/users/explain/colors/colormaps.html). 
    
</div>

In [ ]:
# your answer here
quakes["year"] = pd.to_datetime(quakes["time"]).dt.____

fig, ax = plt.subplots(figsize=(10, 9))


quakes_web = quakes.to_crs(epsg=3857)
nisqually_web = nisqually.to_crs(epsg=3857)

mainshock = nisqually_web[nisqually_web["hours"] == 0]
aftershocks = nisqually_web[(nisqually_web["hours"] > 0) & (nisqually_web["is_earthquake"] == 1)]

quakes_web.plot(ax=ax, 
                column="____",
                cmap="____", 
                markersize=quakes["mag"]**3, # We edited this
                alpha=0.6,
                legend=True
               )
aftershocks.plot(ax=ax, color="red", markersize=25, label="Nisqually aftershocks (first week)")
mainshock.plot(ax=ax, color="gold", edgecolor="black", marker="*", markersize=400, label = "Nisqually mainshock")

cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas)
ax.set_axis_off()
ax.legend(loc="lower left")
ax.set_title("The 2001 Nisqually earthquake and\n 26 years of Puget Sound earthquakes");

Overall, the years are mixed together: about 200-250 M2+ earthquakes every five years, with no corner that "turned on" recently.

The clusters are different, however! Near **Bremerton**, 21 earthquakes happened in 2017 alone, and **Mt. Rainier's** cluster has bursts in 2011 (15 earthquakes) and 2025 (8 earthquakes). A burst of earthquakes in one place with no big mainshock is called a **swarm**.
> The colors are hard to tell apart, so use `.explore()` with `column="year"` and you'll be able to click through them all.

---
## Part 4 : clip a big map to your area
The geology file is the USGS's compilation of the **whole state's geologic map**:

In [ ]:
geology = gpd.read_file(maps + "wa_geology.zip")

print(len(geology), "polygons")
geology.head()

|**column**|**what it holds**|
|:--|:--|
|`ORIG_LABEL`|the unit's label on the original map|
|`GENERALIZE`|a broad rock class: volcanic, sedimentary, unconsolidated...|
|`unit_name`, `unit_age`, `rocktype1`|the unit's name, age, and main rock type|

<span style="font-size:0.85em;color:#6b7a8d;"><em>Data: Horton, J.D., San Juan, C.A., and Stoeser, D.B., 2017, The State Geologic Map Compilation (SGMC) geodatabase of the conterminous United States: U.S. Geological Survey Data Series 1052, Washington polygons, faults, and unit table. In western Washington it is mostly the 1:500,000 Geologic Map of Washington (Hunting and others, 1961).</em></span>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

geology.plot(ax=ax, column="GENERALIZE", cmap="tab20", legend=True, legend_kwds={"loc":"upper left", "bbox_to_anchor": (1,1)})
ax.set_title("Washington geology, by rock class");

### Under the hood: what *is* a shape?
Before we cut this map up, let's look inside `geometry` to see what's going on. You can pull out one polygon the same way you'd pull out one value from any column:

In [ ]:
one_unit = geology.geometry.iloc[0]
one_unit

#### Jupyter **draws** a single shape when it's the last thing in a cell. 
Underneath, a polygon is just a list of `(x, y)` corners, in order that closes back on itself. Here, it's a longitude and a latitude. A point is one `(x, y)` pair.

##### Let's learn about this shape.
First, let's figure out what type of shape it is and what geologic unit it is.

In [ ]:
print(one_unit.geom_type)
print(geology["unit_name"].iloc[0])

Now, let's see how many points it has around it's outisde (how many corners) and the coordiantes of those points.

In [ ]:
corners = list(one_unit.exterior.coords)
print(len(corners), "corners")
print(corners[:5])

##### What about a shape from `quakes`?
First, let's grab the first object:

In [ ]:
one_eq = quakes.geometry.iloc[0]
one_eq

In [ ]:
print(one_eq)

In [ ]:
print(one_eq.x, one_eq.y)

Everything GeoPandas does (area, clip, joins, distances) is math on coordinates, teh same kind of math you've been doing with NumPy arrays all semester.

#### This is the whole state of Washington, but we really only need the Puget sound corner.
- `gpd.clip()` cuts a layer down to an area, keeping only the parts that fall inside.

The easiest area to use is the edges of another layer (remember `.total_bounds` from last lecture):

In [ ]:
area = quakes.total_bounds           # this is the box  around all the earthquakes (west, south, east, north).
print(area)

geology_ps = gpd.clip(geology, area) # NEW: cuts every polygon at the edges of that box, drops the ones outside it
print(len(geology_ps), "polygons left")

#### Clipping is a **mask for shapes**: keep what's inside, throw away the rest.

<div class="alert alert-info">

### Exercsise 4 : geology and faults under the earthquakes

The faults from the same state map are in the `wa_faults` shapefile folder. Read them, clip them to `area`, and make a map with the clipped geology (colored by `GENERALIZE`) on the bottom, the faults as black lines, and `quakes` as small black dots on top.
    
</div>

In [ ]:
faults = gpd.read_file(maps + "wa_faults/____")
faults_ps = gpd.clip(____, area)

fig, ax = plt.subplots(figsize=(8,8))
geology_ps.plot(ax=ax, column="____", cmap="tab20", legend=True, 
                legend_kwds={"loc":"upper left", "bbox_to_anchor": (1,1)})
faults_ps.plot(ax=ax, color="black", linewidth=1.5)
____.plot(ax=ax, color="black", markersize=2)
ax.set_title("Puget Sound geology, mapped faults, and earthquakes");

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

```python
faults = gpd.read_file(maps + "wa_faults/wa_faults.shp")
faults_ps = gpd.clip(faults, area)

fig, ax = plt.subplots(figsize=(8,8))
geology_ps.plot(ax=ax, column="GENERALIZE", cmap="tab20", legend=True, 
                legend_kwds={"loc":"upper left", "bbox_to_anchor": (1,1)})
faults_ps.plot(ax=ax, color="black", linewidth=1.5)
faults_ps.plot(ax=ax, color="black", markersize=2)
ax.set_axis_off()
ax.set_title("Puget Sound geology, mapped faults, and earthquakes");
```
</details>

#### Where is the Seattle Fault?
The **Seattle Fault** runs east-west right under downtown Seattle. it caused a large earthquake about 1100 years ago. **Do you see it on the map?**

It isn't there. Only 16 short faults survive the clip, none under Seattle. This statem map was drawn in 1961, and the evidence that the Seattle Fault is active (a sudden uplift of the shoreline about 1100 years ago) was published in 1992 (Bucknam et al., 1992). Today's best fault map for this kind of question is the USGS **Quaternary Fault and Fold Database**.

---
## Part 5 : spatial joins

In Nashvile, `sjoin` told us which flood zone each gauge was in. The same tool can tell us which rock unit is under each earthquake.

### Which rock unit is under each earthquake?

We can do the same `sjoin` with 1,112 points and 767 polygons:

In [ ]:
quakes_geo = gpd.sjoin(quakes, geology_ps, predicate="within")

print(len(quakes_geo), "earthquakes matched")
quakes_geo[["time", "id", "mag", "depth", "GENERALIZE", "unit_name"]].head()

Okay, So this makes it so that every earthquake now has the columns of the geology unit it sits on.

**Six earthquakes didn't match any polygon**: three sit exactly on the edge of our clip box (we drew the box through the outermost earthquakes), and three fall in tiny gaps between polygons. The default inner join quietly dropped them, so it's always worth checking `len()` before or after a join.

From here it's an ordinary DataFrame, so everything from pandas works, like counting with `groupby`

<div class="alert alert-info">

### Exercise 5 : counting earthquakes by rock class

1. Use `groupby` on `GENERALIZE` and `.size()` to count the earthquakes in each rock class. Sort from most to fewest.
2. A big rock class will catch more earthquakes just because it's big. Find the **area** of each rock class: convert `geology_ps` to UTM zone 10N (EPSG:32610, Washington's UTM zone), add an `area_km2` column, and sum it by `GENERALIZE`.
3. Divide to get **earthquakes per 1,000 km²** for each rock class.

    
</div>

In [ ]:
# 1
counts = quakes_geo.groupby("___").size()
counts = counts.sort_values(ascending=False)
print(counts)

# 2
geology_utm = geology_ps.to_crs(epsg=___)
geology_utm["area_km2"] = geology_utm.___ / 1e6
areas = geology_utm.groupby("GENERALIZE")["area_km2"].sum()

# 3
per_1000_km2 = ___ / ___ * 1000
print(per_1000_km2.round(1))

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

```python
# 1
counts = quakes_geo.groupby("GENERALIZE").size()
counts = counts.sort_values(ascending=False)
print(counts)

# 2
geology_utm = geology_ps.to_crs(epsg=32610)
geology_utm["area_km2"] = geology_utm.area / 1e6
areas = geology_utm.groupby("GENERALIZE")["area_km2"].sum()

# 3
per_1000_km2 = counts / areas * 1000
print(per_1000_km2.round(1))
```
---
```
GENERALIZE
Unconsolidated, undifferentiated                 584
Igneous, volcanic                                214
Water                                            202
Sedimentary, clastic                              72
Metamorphic and Sedimentary, undifferentiated     27
Igneous, intrusive                                 3
Igneous, undifferentiated                          2
Metamorphic, sedimentary                           1
Metamorphic, volcanic                              1
dtype: int64
GENERALIZE
Igneous, intrusive                               21.8
Igneous, undifferentiated                        66.4
Igneous, volcanic                                60.2
Metamorphic and Sedimentary, undifferentiated    43.9
Metamorphic, sedimentary                          8.9
Metamorphic, volcanic                            20.6
Sedimentary, clastic                             62.8
Unconsolidated, undifferentiated                 54.2
Water                                            56.4
dtype: float64
```

By count, "Unconsolidated" (mostly glacial drift) wins easily: 584 earthquakes. But it's also by far the biggest class. Per area, the large classes all land around **54–62 earthquakes per 1,000 km²**, even "Water" (Puget Sound itself). The classes with only a few earthquakes are too small to say much about. (Dividing two Series lines them up by their index, here the rock class, the same way columns lined up in Lecture 16.)


<div class="alert alert-warning">

**Question:** Per area, the earthquakes don't seem to care what's at the surface, not even whether it's water. Why not? (Look at the `depth` column. How deep is the glacial drift?)

</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

The surface map shows what's at the **top**: glacial sediment is tens to hundreds of meters thick, but most of these earthquakes are 10–30 km down, far below it. A geologic map is a picture of the surface, and a spatial join can only match the shapes you give it. Whether the answer means anything is your call, not GeoPandas's. A better question for a surface map is about **shaking**: soft glacial and river deposits shake harder than bedrock, which is one reason parts of Seattle and Olympia were hit hard in 2001.

</details>

---
## Part 6 : buffers: what's within 50 km?

In Lab 06, the Nisqually file held every event **within 50 km** of the epicenter. `.buffer(distance)` draws that circle: every place within a distance of a shape.

In [ ]:
mainshock = nisqually[nisqually["hours"] == 0]

mainshock_utm = mainshock.to_crs(epsg=32610)
circle = mainshock_utm.buffer(50_000)

print(circle.area / 1e6, "km2")

<div class="alert alert-info">

### Exercise 6 : 26 years inside Lab 06's circle

1. Convert `quakes` to EPSG:32610, and use `.within(circle.iloc[0])` to make a mask of the earthquakes inside the circle. How many are there?
2. What's their median depth? How does that compare with Nisqually's 52 km?

</div>

In [ ]:
quakes_utm = quakes.to_crs(epsg=___)

inside = quakes_utm.within(circle.___[0])
print(inside.___(), "earthquakes within 50 km, 2000-2025")

print("median depth:", quakes_utm[___]["depth"].median(), "km")

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

```python
quakes_utm = quakes.to_crs(epsg=32610)

inside = quakes_utm.within(circle.iloc[0])
print(inside.sum(), "earthquakes within 50 km, 2000-2025")

print("median depth:", quakes_utm[inside]["depth"].median(), "km")
```
---
```
277 earthquakes within 50 km, 2000-2025
median depth: 24.49 km
```

277 M2+ earthquakes in 26 years (that includes the mainshock), with a median depth of about 24 km, so most are about half as deep as Nisqually. `.within()` gives a boolean mask, one `True`/`False` per row, so `np.sum`, `.sum()`, and mask selection all work exactly as in Lecture 15.


#### To see the circle, plot it like any other polygon:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
geology_ps.to_crs(epsg=32610).plot(ax=ax, column="GENERALIZE", cmap="tab20", alpha=0.6)
circle.boundary.plot(ax=ax, color="red", linewidth=2)
quakes_utm[inside].plot(ax=ax, color="red", markersize=6)
quakes_utm[~inside].plot(ax=ax, color="black", markersize=3)
mainshock_utm.plot(ax=ax, color="gold", edgecolor="black", marker="*", markersize=400)
ax.set_title("Lab 06's 50 km circle around Nisqually");

### How many earthquakes near **you**?
Let's turn those steps into a **function**, so we can ask the question for any place and any distances. We want to make a reusable thing that:
- makes a point
- converts it to meters
- buffers it
- counts what's inside with a mask

In [ ]:
def quakes_near(lon, lat, km):
    """Count the M2+ earthquakes (2000-2025) within km of a point."""
    here = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]), crs="EPSG:4326")
    circle = here.to_crs(epsg=32616)
    circle_buff = circle.buffer(km * 1000)
    inside = quakes.to_crs(epsg=32616).within(circle_buff.iloc[0])
    return inside

places = {"Seattle (Pike Place Market)": (-122.342, 47.609),
          "Tacoma": (-122.444, 47.253),
          "Olympia": (-122.900, 47.038)}

for name, (lon, lat) in places.items():
    print(f"{name}: {quakes_near(lon, lat, 25)[0].sum()} earthquakes within 25 km")

This loop uses `.items()` to unpack each place's name and it's `(lon, lat)` tuple.

<div class="alert alert-info">

### Exercise 7 : pick your place
1. Add a place of your own to `places` within the region (Mt. Rainier, a neighborhood, an island). Look up its coordinates, and watch the order.
2. Try a different distance: 10 km? 50 km?
3. Where in the Puget SOund would you least like to live, at least by this measure? What does this count leave out about earthquake **hazard**?
    
</div>

In [ ]:
places["____"] = (____, ____)

for name, (lon, lat) in places.items():
    print(f"{name}: {quakes_near(lon, lat, ____)} earthquakes within ____ km")

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

```python
places["Mt. Rainier summit"] = (-121.760, 46.853)

for name, (lon, lat) in places.items():
    print(f"{name}: {quakes_near(lon, lat, 10)[0].sum()} earthquakes within 10 km")
```
---
```
Seattle (Pike Place Market): 17 earthquakes within 10 km
Tacoma: 22 earthquakes within 10 km
Olympia: 2 earthquakes within 10 km
Mt. Rainier summit: 55 earthquakes within 10 km
```

In [ ]:
inside = quakes_near(lon, lat, ____)

fig, ax = plt.subplots(figsize=(7, 7))
geology_ps.to_crs(epsg=32610).plot(ax=ax, column="GENERALIZE", cmap="tab20", alpha=0.6)
quakes_utm[inside].plot(ax=ax, color="red", markersize=6)
quakes_utm[~inside].plot(ax=ax, color="black", markersize=3)
ax.set_axis_off()
ax.set_title("Mt. Rainier earthquakes");

Mt. Rainier wins easily at short distances, because of its swarm of small earthquakes.

But a count of M2+ earthquakes isn't the same as **hazard**: it leaves out how big earthquakes could get, how deep they are (deep ones like Nisqually shake a wide area more gently), what the ground is made of (soft sediment shakes harder), and the faults that are quiet now but can produce rare large earthquakes, like the Seattle Fault. A quiet map can still be a dangerous place.

---
## A useful table: GeoPandas so far

|**you want**|**write**|
|:--|:--|
|points from a table|`gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")`|
|an interactive map|`gdf.explore(column="COL", tooltip=["A", "B"], tiles=cx.providers.Esri.WorldGrayCanvas)`|
|one shape out of the table|`gdf.geometry.iloc[0]`|
|a point's coordinates|`gdf.geometry.x`, `gdf.geometry.y`|
|a different size for every point|`gdf.plot(markersize=gdf["mag"] ** 3)`|
|cut a layer to a box or a shape|`gpd.clip(gdf, other.total_bounds)`|
|which polygon is each point in?|`gpd.sjoin(points, polygons, predicate="within", how="left")`|
|count by a joined column|`joined.groupby("COL").size()`|
|everything within a distance (projected CRS!)|`gdf.to_crs(epsg=32610).buffer(50_000)`|
|inside / outside a shape (a mask)|`gdf.within(shape)`|
|subtract, overlap, or combine shapes|`gpd.overlay(a, b, how="difference")` (or `"intersection"`, `"union"`)|

**For your project:** if your data has a latitude and a longitude, you can now put it on a map, explore it, and ask which polygon (watershed, county, rock unit, flood zone) each point falls in.